# TinyLLM — MIDI Melody Generation

Same tiny transformer architecture, but trained on **music** instead of text.

The model learns musical patterns (scales, rhythms, phrase structures) from
a collection of folk melodies, then generates new ones that you can listen to!

**How it works:** Notes and durations are encoded as tokens, just like words in text.
The model predicts "what note comes next?" the same way the Q&A model predicts "what word comes next?"

**Requirements:** Colab with GPU (Runtime → Change runtime type → T4)

## 1. Setup

In [ ]:
!git clone https://github.com/aadimiro/tinyllm.git
%cd tinyllm
!pip install -q sentencepiece pretty_midi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Train on Melodies

Downloads ~1000 folk melodies (Nottingham dataset), tokenizes them into
musical events, and trains the model to predict the next note.

On a T4 GPU this takes ~5-10 minutes.

In [ ]:
%cd /content/tinyllm/demos/midi
!python midi_train.py

## 3. Generate Melodies!

The model generates token sequences, which we convert back to MIDI files.
You can download and play them in any media player.

In [ ]:
# Generate 5 melodies with different "temperatures"
# Low temperature (0.5) = conservative, sticks to common patterns
# High temperature (1.2) = creative, more surprising
!python midi_generate.py --num 5 --temperature 0.8 --tempo 120

## 4. Listen to the Results

Play the generated MIDI files directly in the notebook using `pretty_midi` + audio synthesis.

In [ ]:
import pretty_midi
import IPython.display as ipd
from pathlib import Path

def play_midi(path, sr=22050):
    """Synthesize and play a MIDI file in the notebook."""
    midi = pretty_midi.PrettyMIDI(str(path))
    audio = midi.synthesize(fs=sr)
    print(f"\n🎵 {path.name} ({midi.get_end_time():.1f}s, "
          f"{len(midi.instruments[0].notes)} notes)")
    return ipd.Audio(audio, rate=sr)

# Play the first generated melody
midi_files = sorted(Path("output").glob("*.mid"))
if midi_files:
    play_midi(midi_files[0])
else:
    print("No MIDI files found. Run generation cell above first.")

In [ ]:
# Play all generated melodies
for f in midi_files:
    display(play_midi(f))

## 5. Experiment with Generation Parameters

Try different settings to hear how they affect the music:

In [ ]:
import sys
sys.path.insert(0, '/content/tinyllm')
sys.path.insert(0, '/content/tinyllm/demos/midi')

from midi_generate import load_midi_model, generate_melody, tokens_to_midi_file, print_melody

device = "cuda" if torch.cuda.is_available() else "cpu"
model, _ = load_midi_model("checkpoints/best.pt", device)

# Conservative: sticks to learned patterns
print("\n=== Conservative (temperature=0.4) ===")
tokens = generate_melody(model, temperature=0.4, top_k=10, device=device)
print_melody(tokens)
tokens_to_midi_file(tokens, "output/conservative.mid", tempo=100)
display(play_midi(Path("output/conservative.mid")))

# Balanced
print("\n=== Balanced (temperature=0.8) ===")
tokens = generate_melody(model, temperature=0.8, top_k=20, device=device)
print_melody(tokens)
tokens_to_midi_file(tokens, "output/balanced.mid", tempo=120)
display(play_midi(Path("output/balanced.mid")))

# Creative: more surprising choices
print("\n=== Creative (temperature=1.2) ===")
tokens = generate_melody(model, temperature=1.2, top_k=40, device=device)
print_melody(tokens)
tokens_to_midi_file(tokens, "output/creative.mid", tempo=140)
display(play_midi(Path("output/creative.mid")))

## 6. Visualize a Melody (Piano Roll)

See the notes plotted on a piano roll — x-axis is time, y-axis is pitch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_piano_roll(midi_path, title="Generated Melody"):
    """Plot a piano roll visualization of a MIDI file."""
    midi = pretty_midi.PrettyMIDI(str(midi_path))
    notes = midi.instruments[0].notes

    fig, ax = plt.subplots(figsize=(14, 4))

    for note in notes:
        ax.barh(note.pitch, note.end - note.start, left=note.start,
                height=0.8, color='steelblue', edgecolor='navy', linewidth=0.5)

    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('MIDI Pitch')
    ax.set_title(title)

    # Add note names on y-axis
    note_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    pitches = [n.pitch for n in notes]
    if pitches:
        ymin, ymax = min(pitches) - 2, max(pitches) + 2
        ax.set_ylim(ymin, ymax)
        ticks = range(ymin, ymax + 1, 2)
        ax.set_yticks(list(ticks))
        ax.set_yticklabels([f"{note_names[p%12]}{p//12-1}" for p in ticks])

    plt.tight_layout()
    plt.show()

# Plot the first generated melody
if midi_files:
    plot_piano_roll(midi_files[0], f"Generated: {midi_files[0].name}")

## 7. Download MIDI Files

Download generated melodies to play in a DAW, GarageBand, or any MIDI player.

In [ ]:
from google.colab import files
import shutil

# Zip all generated melodies
shutil.make_archive('generated_melodies', 'zip', 'output')
files.download('generated_melodies.zip')
print("Download started! Open the .mid files in any media player or DAW.")